In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
df = pd.read_csv("myntra_products_catalog.csv")
df.head()


,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [3]:
df.shape

(12491, 8)

In [4]:
df.columns

Index(['ProductID', 'ProductName', 'ProductBrand', 'Gender', 'Price (INR)',
       'NumImages', 'Description', 'PrimaryColor'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12491 entries, 0 to 12490
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ProductID     12491 non-null  int64 
 1   ProductName   12491 non-null  object
 2   ProductBrand  12491 non-null  object
 3   Gender        12491 non-null  object
 4   Price (INR)   12491 non-null  int64 
 5   NumImages     12491 non-null  int64 
 6   Description   12491 non-null  object
 7   PrimaryColor  11597 non-null  object
dtypes: int64(3), object(5)
memory usage: 780.8+ KB


In [6]:
df.isnull().sum()

,0
ProductID,0
ProductName,0
ProductBrand,0
Gender,0
Price (INR),0
NumImages,0
Description,0
PrimaryColor,894


In [7]:
df = df.fillna("")

In [8]:
df = df.drop_duplicates()

In [9]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [10]:
df["Features"] = (
    df["ProductName"].astype(str) + " " +
    df["ProductBrand"].astype(str) + " " +
    df["Gender"].astype(str) + " " +
    df["Description"].astype(str) + " " +
    df["PrimaryColor"].astype(str)
)
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor,Features
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black,DKNY Unisex Black & Grey Printed Medium Trolle...
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige,EthnoVogue Women Beige & Grey Made to Measure ...
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink,SPYKAR Women Pink Alexa Super Skinny Fit High-...
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue,Raymond Men Blue Self-Design Single-Breasted B...
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White,Parx Men Brown & Off-White Slim Fit Printed Ca...


In [11]:
df[["ProductName", "Features"]].head()

,ProductName,Features
0,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY Unisex Black & Grey Printed Medium Trolle...
1,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue Women Beige & Grey Made to Measure ...
2,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR Women Pink Alexa Super Skinny Fit High-...
3,Raymond Men Blue Self-Design Single-Breasted B...,Raymond Men Blue Self-Design Single-Breasted B...
4,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx Men Brown & Off-White Slim Fit Printed Ca...


In [12]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [13]:
def preprocess(text):
  text = text.lower()
  text = re.sub(r'[^a-z A-Z\s]','',text)
  words = word_tokenize(text)
  words = [word for word in words if word not in stop_words]
  words = [lemmatizer.lemmatize(word) for word in words]
  return " ".join(words)

In [14]:
df["Processed_Features"] = df["Features"].apply(preprocess)
df[["ProductName", "Processed_Features"]].head()

,ProductName,Processed_Features
0,DKNY Unisex Black & Grey Printed Medium Trolle...,dkny unisex black grey printed medium trolley ...
1,EthnoVogue Women Beige & Grey Made to Measure ...,ethnovogue woman beige grey made measure custo...
2,SPYKAR Women Pink Alexa Super Skinny Fit High-...,spykar woman pink alexa super skinny fit highr...
3,Raymond Men Blue Self-Design Single-Breasted B...,raymond men blue selfdesign singlebreasted ban...
4,Parx Men Brown & Off-White Slim Fit Printed Ca...,parx men brown offwhite slim fit printed casua...


In [15]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Processed_Features'])
tfidf_matrix.shape
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 252394 stored elements and shape (12491, 10083)>
  Coords	Values
  (0, 2463)	0.2925841343929048
  (0, 9411)	0.13786133031150657
  (0, 726)	0.1693471849390207
  (0, 3879)	0.14093457780367027
  (0, 6880)	0.10169230867283666
  (0, 5387)	0.19193522508113983
  (0, 9245)	0.41640683596612543
  (0, 497)	0.19691337330128772
  (0, 7613)	0.10651309129054772
  (0, 9284)	0.1436880717674056
  (0, 5099)	0.14494935551573476
  (0, 3988)	0.21221426178960945
  (0, 9147)	0.1646023051920173
  (0, 6047)	0.17254867199285656
  (0, 7870)	0.07845470982568392
  (0, 7319)	0.1436880717674056
  (0, 3504)	0.11239602705119675
  (0, 1844)	0.1430849915655525
  (0, 5639)	0.14494935551573476
  (0, 4415)	0.1484840727237969
  (0, 7966)	0.1484840727237969
  (0, 9805)	0.14030689629068163
  (0, 5231)	0.08792669912021431
  (0, 10059)	0.23850761194719233
  (0, 1708)	0.17809379468235642
  :	:
  (12489, 4209)	0.37593381490686745
  (12489, 5283)	0.3378569021755022
  (12

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_matrix.shape

similarity_df = pd.DataFrame(similarity_matrix)
similarity_df.head()

,0,1,2,3,4,5,6,7,8,9,...,12481,12482,12483,12484,12485,12486,12487,12488,12489,12490
0,1.000000,0.022610,0.021327,0.051552,0.055160,0.024917,0.035328,0.020653,0.000000,0.842454,...,0.016273,0.016938,0.031914,0.025033,0.028584,0.086410,0.006785,0.019470,0.000000,0.067170
1,0.022610,1.000000,0.008412,0.003148,0.010903,0.004734,0.004766,0.008146,0.005332,0.009138,...,0.008436,0.022365,0.008660,0.009113,0.034089,0.004571,0.008485,0.005607,0.000000,0.016919
2,0.021327,0.008412,1.000000,0.051044,0.022112,0.026473,0.023999,0.727716,0.007786,0.020750,...,0.051500,0.021335,0.015232,0.073817,0.005391,0.330570,0.016523,0.025270,0.000000,0.063295
3,0.051552,0.003148,0.051044,1.000000,0.063492,0.066949,0.119643,0.049431,0.071996,0.037802,...,0.011373,0.007622,0.028568,0.073990,0.014708,0.085772,0.011270,0.073878,0.011492,0.025921
4,0.055160,0.010903,0.022112,0.063492,1.000000,0.239220,0.696929,0.021413,0.411688,0.031669,...,0.006622,0.025137,0.006797,0.119623,0.000000,0.063835,0.000000,0.101334,0.022623,0.051025


In [17]:
def recommend_by_brand(brand):

    products = df[df["ProductBrand"].str.lower() == brand.lower()]
    if products.empty:
        print("Brand not found!")
        return
    index = products.index[0]

    similarity_scores = list(enumerate(similarity_matrix[index]))
    similarity_scores = sorted(similarity_scores,
                               key=lambda x: x[1],
                               reverse=True)

    top_products = similarity_scores[1:6]
    print("\nThe Recommended Products\n")

    for i, score in top_products:
        print("Product Name :", df.iloc[i]["ProductName"])
        print("Brand        :", df.iloc[i]["ProductBrand"])
        print("Gender       :", df.iloc[i]["Gender"])
        print("Price (INR)  :", df.iloc[i]["Price (INR)"])
        print("Similarity   :", round(score,2))
        print("-"*60)

In [18]:
df.columns

Index(['ProductID', 'ProductName', 'ProductBrand', 'Gender', 'Price (INR)',
       'NumImages', 'Description', 'PrimaryColor', 'Features',
       'Processed_Features'],
      dtype='object')

In [19]:
df["ProductName"].head(20)

,ProductName
0,DKNY Unisex Black & Grey Printed Medium Trolle...
1,EthnoVogue Women Beige & Grey Made to Measure ...
2,SPYKAR Women Pink Alexa Super Skinny Fit High-...
3,Raymond Men Blue Self-Design Single-Breasted B...
4,Parx Men Brown & Off-White Slim Fit Printed Ca...
5,SHOWOFF Men Brown Solid Slim Fit Regular Shorts
6,Parx Men Blue Slim Fit Checked Casual Shirt
7,SPYKAR Women Burgundy Alexa Super Skinny Fit H...
8,Parx Men Brown Tapered Fit Solid Regular Trousers
9,DKNY Unisex Black Large Trolley Bag


In [23]:
print(" Brands:\n")

brands = sorted(df["ProductBrand"].dropna().unique())

for brand in brands:
    print(brand)

 Brands:

109F
20Dresses
612 league
7Rainbow
AASK
ABELINO
ADIDAS
ADIDAS Originals
ADIVA
ADORENITE
ADORN by Nikita Ladiwala
AHIKA
AIGNER
AKKRITI BY PANTALOONS
AND
ANNA SUI
ANTS
AURELIA
AVANT-GARDE PARIS
AWW HUNNIE
Aapno Rajasthan
Aarika
Aber & Q
Abhishti
AccessHer
Aditi Wasan
Aerosoles
Ahalyaa
Aj DEZInES
Akiva
Alcis
Alena
Alina decor
Allen Cooper
Allen Solly
Allen Solly Junior
Allen Solly Sport
Allen Solly Woman
Alom
Alvaro Castagnino
Amante
American Crew
Anekaant
Annabelle by Pantaloons
Anouk
Antheaa
Archies
Arrow
Arrow New York
Arrow Sport
Athena
Aujjessa
Avira Home
Ayesha
BBLUNT
BEARDO
BEAT LONDON by PEPE JEANS
BERING
BIANCA
BOLLYGLOW
BUKKUM
Baggit
Bamboo Tree Jewels
Barbie by Many Frocks &
Basics
Batman
Be Indi
Being Human
Beli
Belle Fille
Bene Kleed
Besiva
Bhama Couture
Biba
Bitiya by Bhama
Black coffee
Blackberrys
Blacksmith
Blisscovered
Blissta
Blue Giraffe
Blue Saint
Bodycare
Bohemia Crystal
Bonjour
Bossini
BownBee
Braun
Bruno Manetti
Bubblegummers
BuckleUp
Bvlgari
C9 AIRWEAR
CA

In [20]:
print("Total Products:", len(df))

Total Products: 12491


In [22]:
brand = input("Enter Brand Name: ")
recommend_by_brand(brand)

Enter Brand Name: SHOWOFF

The Recommended Products

Product Name : SHOWOFF Men Khaki Solid Slim Fit Regular Shorts
Brand        : SHOWOFF
Gender       : Men
Price (INR)  : 791
Similarity   : 0.67
------------------------------------------------------------
Product Name : SHOWOFF Men Tan Brown Solid Biker Jacket
Brand        : SHOWOFF
Gender       : Men
Price (INR)  : 3698
Similarity   : 0.64
------------------------------------------------------------
Product Name : SHOWOFF Men Grey Printed Slim Fit Regular Shorts
Brand        : SHOWOFF
Gender       : Men
Price (INR)  : 835
Similarity   : 0.61
------------------------------------------------------------
Product Name : SHOWOFF Men Green Printed Bomber Jacket
Brand        : SHOWOFF
Gender       : Men
Price (INR)  : 3698
Similarity   : 0.49
------------------------------------------------------------
Product Name : Indian Terrain Men Brown Brooklyn Slim Fit Solid Regular Trousers
Brand        : Indian Terrain
Gender       : Men
Price (IN